In [3]:
import os
from typing import Dict, List, Union, Any

import pandas as pd
import gseapy as gp
from gseapy.plot import barplot, dotplot

import pickle


def _normalize_gene_list(raw_genes: Any) -> List[str]:
    if raw_genes is None:
        return []
    if isinstance(raw_genes, (set, tuple)):
        raw_genes = list(raw_genes)
    if not isinstance(raw_genes, list):
        # sometimes a single string or something else
        raw_genes = [raw_genes]

    cleaned: List[str] = []
    seen = set()
    for gene in raw_genes:
        gene_symbol = str(gene).strip().upper()
        if gene_symbol == "":
            continue
        if gene_symbol in seen:
            continue
        seen.add(gene_symbol)
        cleaned.append(gene_symbol)
    return cleaned


def resolve_reactome_gene_sets(
    reactome_pickle_file_path: str,
    organism: str = "Human",
) -> Dict[str, List[str]]:
    """
    Load a pickled Reactome pathway gene set object and normalize it into:
        { "PATHWAY_NAME_OR_ID": ["GENE1", "GENE2", ...], ... }

    Supported pickle layouts (examples):
      1) dict: {pathway: [genes]}
      2) dict: {"Human": {pathway: [genes]}, "Mouse": {...}}
      3) dict: {pathway: {"genes": [...]} } or {"members": [...]} etc.
      4) list: [{"pathway": "...", "genes": [...]}, ...] (or similar keys)
    """
    if not os.path.exists(reactome_pickle_file_path):
        raise FileNotFoundError(f"Reactome pickle not found: {reactome_pickle_file_path}")

    with open(reactome_pickle_file_path, "rb") as file_handle:
        loaded_object = pickle.load(file_handle)

    # If top-level has organism key, descend
    if isinstance(loaded_object, dict):
        organism_key_candidates = [
            organism,
            organism.lower(),
            organism.upper(),
            organism.title(),
            "Homo sapiens",
            "HOMO SAPIENS",
            "human",
            "HUMAN",
        ]
        for candidate_key in organism_key_candidates:
            if candidate_key in loaded_object and isinstance(loaded_object[candidate_key], (dict, list)):
                loaded_object = loaded_object[candidate_key]
                break

    pathway_to_genes: Dict[str, List[str]] = {}

    # Case A: dict
    if isinstance(loaded_object, dict):
        # A1) {pathway: [genes]}
        all_values_are_sequences = all(isinstance(value, (list, set, tuple)) for value in loaded_object.values())
        if all_values_are_sequences:
            for pathway_name, genes in loaded_object.items():
                normalized_genes = _normalize_gene_list(genes)
                if len(normalized_genes) == 0:
                    continue
                pathway_to_genes[str(pathway_name)] = normalized_genes

        else:
            # A2) {pathway: {genes: [...]}} or other nested dict formats
            gene_field_candidates = ["genes", "gene_list", "members", "symbols", "geneSymbols", "GENES"]
            name_field_candidates = ["name", "pathway_name", "pathway", "term", "Term", "description"]

            for pathway_key, pathway_value in loaded_object.items():
                pathway_label = str(pathway_key)

                if isinstance(pathway_value, dict):
                    # try to find a nicer pathway label inside
                    for name_field in name_field_candidates:
                        if name_field in pathway_value:
                            pathway_label = str(pathway_value[name_field])
                            break

                    found_gene_list: Optional[Any] = None
                    for gene_field in gene_field_candidates:
                        if gene_field in pathway_value:
                            found_gene_list = pathway_value[gene_field]
                            break

                    normalized_genes = _normalize_gene_list(found_gene_list)
                    if len(normalized_genes) == 0:
                        continue
                    pathway_to_genes[pathway_label] = normalized_genes

                elif isinstance(pathway_value, (list, set, tuple)):
                    normalized_genes = _normalize_gene_list(pathway_value)
                    if len(normalized_genes) == 0:
                        continue
                    pathway_to_genes[pathway_label] = normalized_genes

    # Case B: list of records
    elif isinstance(loaded_object, list):
        pathway_field_candidates = ["pathway", "name", "term", "Term", "pathway_name", "description"]
        gene_field_candidates = ["genes", "gene_list", "members", "symbols", "geneSymbols", "GENES"]

        for record in loaded_object:
            if not isinstance(record, dict):
                continue

            pathway_label: Optional[str] = None
            for pathway_field in pathway_field_candidates:
                if pathway_field in record:
                    pathway_label = str(record[pathway_field])
                    break
            if not pathway_label:
                continue

            raw_genes: Any = None
            for gene_field in gene_field_candidates:
                if gene_field in record:
                    raw_genes = record[gene_field]
                    break

            normalized_genes = _normalize_gene_list(raw_genes)
            if len(normalized_genes) == 0:
                continue
            pathway_to_genes[pathway_label] = normalized_genes

    else:
        raise ValueError(f"Unsupported reactome pickle type: {type(loaded_object)}")

    if len(pathway_to_genes) == 0:
        raise ValueError(
            "Loaded reactome gene sets are empty after normalization. "
            "Check the pickle content/format."
        )

    return pathway_to_genes

def perform_reactome_enrichment_for_cell_type(
    cell_type: str,
    biomarker_directory: str,
    reactome_gene_sets: Union[str, Dict[str, List[str]]],
    top_gene_count: int = 100,
    organism: str = "human",
    plot_top_term: int = 20,
    plot_cutoff: float = 0.05,
) -> str:
    biomarker_file_path = os.path.join(
        biomarker_directory, f"celltype_gene_biomarker_{cell_type}.tsv"
    )
    output_table_path = os.path.join(
        biomarker_directory, f"celltype_gene_biomarker_{cell_type}_gsea.tsv"
    )

    biomarker_dataframe = pd.read_csv(biomarker_file_path, sep="\t")
    if "gene" not in biomarker_dataframe.columns:
        raise ValueError(
            f"'gene' column not found: {biomarker_file_path}. "
            f"Available columns: {list(biomarker_dataframe.columns)}"
        )

    # top N genes (order-preserving, drop duplicates)
    gene_series = biomarker_dataframe["gene"].astype(str).str.strip()
    gene_series = gene_series[gene_series != ""].str.upper()

    seen = set()
    top_genes: List[str] = []
    for gene_symbol in gene_series.tolist():
        if gene_symbol in seen:
            continue
        seen.add(gene_symbol)
        top_genes.append(gene_symbol)
        if len(top_genes) >= top_gene_count:
            break

    if len(top_genes) == 0:
        raise ValueError(f"No valid genes found in {biomarker_file_path}.")

    enrichment_result = gp.enrichr(
        gene_list=top_genes,
        gene_sets=reactome_gene_sets,  # string library name OR dict
        organism=organism,
        outdir=None,   # 우리가 직접 그림 저장할 거라서 None 유지
        no_plot=True,  # enrichr 내부 자동 플롯 비활성화
    )

    # 결과 테이블 저장
    result_dataframe = getattr(enrichment_result, "res2d", None)
    if result_dataframe is None:
        result_dataframe = enrichment_result.results

    result_dataframe.to_csv(output_table_path, sep="\t", index=False)

    # 그림 저장 디렉토리
    plot_directory = os.path.join(biomarker_directory, "gsea_plots", cell_type)
    os.makedirs(plot_directory, exist_ok=True)

    # 유의미한 결과가 없으면 스킵
    if result_dataframe is None or len(result_dataframe) == 0:
        print(f"[warn] No enrichment terms for {cell_type}. Skip plots.")
        return output_table_path

    # barplot / dotplot 저장 (png + pdf)
    title = f"{cell_type} Reactome enrichment"
    barplot_png = os.path.join(plot_directory, f"{cell_type}_reactome_barplot.png")
    barplot_pdf = os.path.join(plot_directory, f"{cell_type}_reactome_barplot.pdf")
    dotplot_png = os.path.join(plot_directory, f"{cell_type}_reactome_dotplot.png")
    dotplot_pdf = os.path.join(plot_directory, f"{cell_type}_reactome_dotplot.pdf")

    # NOTE: cutoff는 "그림에 표시할 term 필터(FDR 등)"로만 작동하고, 테이블 저장에는 영향 없음
    barplot(
        result_dataframe,
        title=title,
        cutoff=plot_cutoff,
        top_term=plot_top_term,
        ofname=barplot_png,
    )
    barplot(
        result_dataframe,
        title=title,
        cutoff=plot_cutoff,
        top_term=plot_top_term,
        ofname=barplot_pdf,
    )

    dotplot(
        result_dataframe,
        title=title,
        cutoff=plot_cutoff,
        top_term=plot_top_term,
        ofname=dotplot_png,
    )
    dotplot(
        result_dataframe,
        title=title,
        cutoff=plot_cutoff,
        top_term=plot_top_term,
        ofname=dotplot_pdf,
    )

    print(f"[saved plots] {plot_directory}")
    return output_table_path


# -------------------------
# Example usage (your case) - 그림까지 저장
# -------------------------
run_directory = "/data2/project/bin_jip/Biomarker/experiment_asthma_find/train_runs/20260222_031424_enc-transformer_conv_emb-concat_pool-attention_clf-mlp_bag32_bpp16_lr0.0003_seed42_split3"
biomarker_directory = os.path.join(run_directory, "biomarker")

reactome_pickle_file_path = "/data2/project/bin_jip/Biomarker/data/pathway/reactome_pathway.pkl"
reactome_gene_sets = resolve_reactome_gene_sets(reactome_pickle_file_path, organism="Human")

cell_types = ["CD14", "CD4_TEM", "B_Memory", "CD4_Naive", "CD4_TCM", "NK"]
for cell_type in cell_types:
    saved_path = perform_reactome_enrichment_for_cell_type(
        cell_type=cell_type,
        biomarker_directory=biomarker_directory,
        reactome_gene_sets=reactome_gene_sets,
        top_gene_count=100,
        organism="human",
        plot_top_term=20,
        plot_cutoff=0.05,
    )
    print(f"[saved table] {saved_path}")


[saved plots] /data2/project/bin_jip/Biomarker/experiment_asthma_find/train_runs/20260222_031424_enc-transformer_conv_emb-concat_pool-attention_clf-mlp_bag32_bpp16_lr0.0003_seed42_split3/biomarker/gsea_plots/CD14
[saved table] /data2/project/bin_jip/Biomarker/experiment_asthma_find/train_runs/20260222_031424_enc-transformer_conv_emb-concat_pool-attention_clf-mlp_bag32_bpp16_lr0.0003_seed42_split3/biomarker/celltype_gene_biomarker_CD14_gsea.tsv
[saved plots] /data2/project/bin_jip/Biomarker/experiment_asthma_find/train_runs/20260222_031424_enc-transformer_conv_emb-concat_pool-attention_clf-mlp_bag32_bpp16_lr0.0003_seed42_split3/biomarker/gsea_plots/CD4_TEM
[saved table] /data2/project/bin_jip/Biomarker/experiment_asthma_find/train_runs/20260222_031424_enc-transformer_conv_emb-concat_pool-attention_clf-mlp_bag32_bpp16_lr0.0003_seed42_split3/biomarker/celltype_gene_biomarker_CD4_TEM_gsea.tsv
[saved plots] /data2/project/bin_jip/Biomarker/experiment_asthma_find/train_runs/20260222_031424_e

In [5]:
# pathway names 중 기준 넘은 것들 출력

for cell_type in cell_types:
    table_path = os.path.join(
        biomarker_directory, f"celltype_gene_biomarker_{cell_type}_gsea.tsv"
    )
    if not os.path.exists(table_path):
        print(f"[warn] Result table not found for {cell_type}: {table_path}")
        continue

    df = pd.read_csv(table_path, sep="\t")
    significant_terms = df[df["Adjusted P-value"] < 0.05]
    print(f"\n=== {cell_type} significant pathways (FDR < 0.05) ===")
    if len(significant_terms) == 0:
        print("No significant pathways found.")
    else:
        for idx, row in significant_terms.iterrows():
            print(f"{row['Term']} (Adj P-value: {row['Adjusted P-value']:.4e})")





=== CD14 significant pathways (FDR < 0.05) ===
REACTOME_ACTIVATED_TAK1_MEDIATES_P38_MAPK_ACTIVATION (Adj P-value: 5.0319e-07)
REACTOME_ACTIVATION_OF_NF_KAPPAB_IN_B_CELLS (Adj P-value: 3.8109e-02)
REACTOME_ADIPOGENESIS (Adj P-value: 6.3555e-07)
REACTOME_ADVANCED_GLYCOSYLATION_ENDPRODUCT_RECEPTOR_SIGNALING (Adj P-value: 3.1807e-05)
REACTOME_ALPHA_PROTEIN_KINASE_1_SIGNALING_PATHWAY (Adj P-value: 1.3686e-02)
REACTOME_ATF4_ACTIVATES_GENES_IN_RESPONSE_TO_ENDOPLASMIC_RETICULUM_STRESS (Adj P-value: 7.0057e-03)
REACTOME_CD209_DC_SIGN_SIGNALING (Adj P-value: 4.2780e-02)
REACTOME_CELLULAR_SENESCENCE (Adj P-value: 4.9049e-04)
REACTOME_CHEMOKINE_RECEPTORS_BIND_CHEMOKINES (Adj P-value: 7.0057e-03)
REACTOME_CLEC7A_DECTIN_1_SIGNALING (Adj P-value: 4.4353e-10)
REACTOME_CO_INHIBITION_BY_PD_1 (Adj P-value: 1.3180e-06)
REACTOME_CYTOSOLIC_SENSORS_OF_PATHOGEN_ASSOCIATED_DNA (Adj P-value: 1.0203e-02)
REACTOME_C_TYPE_LECTIN_RECEPTORS_CLRS (Adj P-value: 2.5885e-08)
REACTOME_DDX58_IFIH1_MEDIATED_INDUCTION_OF_I